In [1]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Default paramaters
SEED = 460
S0 = 100
K = 95
R = 0.03
Q = 0.02
SIGMA = 0.25
M = 50
T = 0.5
NUM_PATH = 20_000

# Question 1


In [2]:
# Question 1 functions


def GBM(
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate GBM with annualized paramaters
    """

    # Set seed for reproducibility
    rng = np.random.default_rng(seed=seed)
    dt = terminal_time / time_steps

    # Generate all brownian motion paths using standard normal distribution
    Z = rng.standard_normal((time_steps, num_path))

    # Compute log returns for each path and for each time step
    drift = (interest_rate - dividend_yield - 0.5 * volatility**2) * dt
    diffusion = volatility * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    # Cumlate the log returns to get the log stock price
    log_prices = np.vstack([np.zeros(num_path), np.cumsum(log_returns, axis=0)])

    Ws = np.cumsum(np.vstack([np.zeros(num_path), np.sqrt(dt) * Z]), axis=0)

    # Convert log prices to actual stock prices
    stock_prices = initial_stock_price * np.exp(log_prices)

    return stock_prices, Ws


def apply_discount_factor(price, interest_rate=R, terminal_time=T):
    """
    Discount the price of an option to present value
    """
    return price * np.exp(-interest_rate * terminal_time)


def Undiscounted_Asian_Put(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate undiscounted payoff for Asian put option where stock prices are modeled using GBM
    """
    # Compute the stock price paths using GBM
    stock_prices, _ = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the payoff of the Asian put option for each path
    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    return payoffs


def Monte_Carlo_Simulation(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    put_prices = apply_discount_factor(put_payoffs, interest_rate, terminal_time)

    return put_prices, np.mean(put_prices)

In [3]:
# Question 1 example usage
put_prices, average_put_price = Monte_Carlo_Simulation()
print(f"Average Asian Put Price: {average_put_price:.4f}")

Average Asian Put Price: 1.8680


# Question 5


In [4]:
# Question 5 functions

# Finite difference step size 1 dollar in initial stock price
STOCK_PRICE_DELTA = 1

# Finite difference step size 1%
VOLATILITY_DELTA = 0.01


# Helper function to estimate the mean and variance of a given greek path
def estimate_variance(greek_path, num_path=NUM_PATH):
    return np.mean(greek_path), np.var(greek_path, ddof=1) / num_path


# Finite Difference Methods
def finite_difference_method_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=STOCK_PRICE_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the delta of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price + h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the price of the Asian put option at S0 - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price - h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the delta using finite difference method
    delta_path = apply_discount_factor(
        (plus_h_put_payoffs - minus_h_put_payoffs) / (2 * h),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(delta_path, num_path)


def finite_difference_method_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=VOLATILITY_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the vega of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 with volatility + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility + h,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the price of the Asian put option at S0 with volatility - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility - h,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the vega using finite difference method
    vega_path = apply_discount_factor(
        (plus_h_put_payoffs - minus_h_put_payoffs) / (2 * h),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(vega_path, num_path)


def finite_difference_method_gamma(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    h=STOCK_PRICE_DELTA,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Compute the gamma of an Asian put option using finite difference method
    """

    # Compute the price of the Asian put option at S0 + h
    plus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price + h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the price of the Asian put option at S0 - h
    minus_h_put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price - h,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the price of the Asian put option at S0
    put_payoffs = Undiscounted_Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the gamma using finite difference method
    gamma_path = apply_discount_factor(
        (plus_h_put_payoffs - 2 * put_payoffs + minus_h_put_payoffs) / (h**2),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(gamma_path, num_path)


# Pathwise Estimators
def pathwise_estimator_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Generate stock price paths
    stock_prices, _ = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    indicators = (np.mean(stock_prices[1:, :], axis=0) < strike_price).astype(float)

    delta_path = apply_discount_factor(
        (
            (-1)
            * indicators
            * np.mean(stock_prices[1:, :], axis=0)
            / initial_stock_price
        ),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(delta_path, num_path)


def pathwise_estimator_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Generate stock price and Brownian motion paths
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    indicators = (np.mean(stock_prices[1:, :], axis=0) < strike_price).astype(float)

    times = np.linspace(0, terminal_time, time_steps + 1)[:, None]

    vega_path = apply_discount_factor(
        -indicators
        * np.mean(
            stock_prices[1:, :] * (Ws[1:, :] - volatility * times[1:, :]), axis=0
        ),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(vega_path, num_path)


def likelihood_estimator_delta(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Generate stock price and Brownian motion paths
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    delta_path = apply_discount_factor(
        payoffs * Ws[1, :] / (initial_stock_price * volatility * dt),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(delta_path, num_path)


def likelihood_estimator_vega(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Generate stock price and Brownian motion paths
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    dW = np.diff(Ws, axis=0)

    vega_path = apply_discount_factor(
        payoffs
        * (
            -time_steps / volatility
            + np.sum(dW**2, axis=0) / (volatility * dt)
            - Ws[-1, :]
        ),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(vega_path, num_path)


def likelihood_estimator_gamma(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    # Generate stock price and Brownian motion paths
    stock_prices, Ws = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    dt = terminal_time / time_steps

    average_prices = np.mean(stock_prices[1:, :], axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    L_s0 = Ws[1, :] / (initial_stock_price * volatility * dt)

    gamma_path = apply_discount_factor(
        payoffs
        * (
            L_s0**2
            - L_s0 / initial_stock_price
            - 1 / (initial_stock_price**2 * volatility**2 * dt)
        ),
        interest_rate,
        terminal_time,
    )

    return estimate_variance(gamma_path, num_path)


def compare_estimators(
    methods, references, num_experiment, num_path=NUM_PATH, seed=SEED
):
    """
    Compare mean variances of given methods with reference methods.
    """

    results = []

    for name, (greek, estimator) in methods.items():

        estimate = []
        variance = []

        for i in range(num_experiment):
            est, var = estimator(num_path=num_path, seed=seed + i)
            estimate.append(est)
            variance.append(var)

        mean_estimate = np.mean(estimate)
        var_estimate = np.var(estimate, ddof=1)
        mean_var_experiment = np.mean(variance)

        bias = mean_estimate - references[greek]
        mse = var_estimate + bias**2

        results.append(
            {
                "Method": name,
                "Greek": greek,
                "Mean Estimate": mean_estimate,
                "Variance of Estimate": var_estimate,
                "Mean Variance of Experiment": mean_var_experiment,
                "Bias": bias,
                "MSE": mse,
            }
        )

    return results

Here we can prove that both the pathwise method and likelihood ratio method are unbiased estimators for the greeks. So we uses the likelihood ratio estimator as our true param, and comparing other method with that result to obtain bias.


In [ ]:
methods = {
    "FD Delta": ("Delta", finite_difference_method_delta),
    "PW Delta": ("Delta", pathwise_estimator_delta),
    "LRM Delta": ("Delta", likelihood_estimator_delta),
    "FD Vega": ("Vega", finite_difference_method_vega),
    "PW Vega": ("Vega", pathwise_estimator_vega),
    "LRM Vega": ("Vega", likelihood_estimator_vega),
    "FD Gamma": ("Gamma", finite_difference_method_gamma),
    "LRM Gamma": ("Gamma", likelihood_estimator_gamma),
}

# We use LRM as our reference method for Delta, Vega and Gamma with following config
N_REF = 1_000_000
REF_SEED = 459

lrm_delta, lrm_delta_var = likelihood_estimator_delta(num_path=N_REF, seed=REF_SEED)

lrm_vega, lrm_vega_var = likelihood_estimator_vega(num_path=N_REF, seed=REF_SEED)

lrm_gamma, lrm_gamma_var = likelihood_estimator_gamma(num_path=N_REF, seed=REF_SEED)

print("Likelihood Delta:", lrm_delta, "SE:", np.sqrt(lrm_delta_var))
print("Likelihood Vega:", lrm_vega, "SE:", np.sqrt(lrm_vega_var))
print("Likelihood Gamma:", lrm_gamma, "SE:", np.sqrt(lrm_gamma_var))

Delta reference: -0.2832037342535516 SE: 0.0017928945758564749
Vega reference: 14.11430681931831 SE: 0.17894397594282171
Gamma reference: 0.032076246455423146 SE: 0.0011018305430063788


Using the Likelihood method, the vega's se is high, we compare it with pathwise variance, we also compare it with Pathwise delta.


In [ ]:
pw_vega, pw_vega_var = pathwise_estimator_vega(num_path=N_REF, seed=REF_SEED)
pw_delta, pw_delta_var = pathwise_estimator_delta(num_path=N_REF, seed=REF_SEED)

print("PW Vega:", pw_vega, "SE:", np.sqrt(pw_vega_var))
print("PW Delta:", pw_delta, "SE:", np.sqrt(pw_delta_var))

PW Vega reference: 13.829438256014248 SE: 0.02170409261769095
PW Delta reference: -0.28369692254414003 SE: 0.0004110888842314087


It seems like the PW has a lower variance than likelihood method, so we will use PW vega and delta as reference and use likelihood method's gamma for reference.


In [ ]:
NUM_EXPERIMENT = 100

references = {
    "Delta": pw_delta,
    "Vega": pw_vega,
    "Gamma": lrm_gamma,
}


results = compare_estimators(
    methods=methods,
    references=references,
    num_experiment=NUM_EXPERIMENT,
)

comparison_df = pd.DataFrame(results)

display(comparison_df)

,Method,Greek,Mean Estimate,Variance of Estimate,Mean Variance of Experiment,Bias,MSE
0,FD Delta,Delta,-0.283132,7.977123e-06,7.936062e-06,0.000565,0.000008
1,PW Delta,Delta,-0.282774,8.583137e-06,8.436722e-06,0.000923,0.000009
2,LRM Delta,Delta,-0.283024,1.587923e-04,1.601121e-04,0.000673,0.000159
3,FD Vega,Vega,13.767618,2.383971e-02,2.337928e-02,-0.061820,0.027661
4,PW Vega,Vega,13.770400,2.400092e-02,2.344997e-02,-0.059038,0.027486
5,LRM Vega,Vega,13.750319,1.588899e+00,1.580587e+00,-0.079119,1.595159
6,FD Gamma,Gamma,0.032687,9.980164e-07,9.666054e-07,0.000610,0.000001
7,LRM Gamma,Gamma,0.032718,6.621180e-05,6.103676e-05,0.000642,0.000067
